In [1]:
import pandas as pd 
import numpy as np
import re
from typing import List, Dict, Any
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEndpointEmbeddings
import PyPDF2
from pathlib import Path
import os

/Users/nitpatel/Coding/savan Bhai /project_1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from dotenv import load_dotenv
load_dotenv()   
import os

groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(api_key=groq_api_key, model_name="meta-llama/llama-4-maverick-17b-128e-instruct")

In [3]:
# Import PDF processing libraries
import PyPDF2
import fitz
from pathlib import Path
import os

# Function to extract text from PDF

def extract_text_from_pdf(pdf_path):
    """Extract text from PDF using PyMuPDF"""
    try:
        doc = fitz.open(pdf_path)
        text = ""
        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            text += page.get_text()
        doc.close()
        return text
    except Exception as e:
        print(f"Error reading {pdf_path}: {e}")
        return None

# Function to ingest all PDFs from data folder
def ingest_pdfs_from_data_folder(data_folder="data"):
    """Ingest all PDF files from the data folder"""
    pdf_documents = []
    data_path = Path(data_folder)
    
    if not data_path.exists():
        print(f"Data folder {data_folder} not found")
        return pdf_documents
    
    # Find all PDF files
    pdf_files = list(data_path.glob("*.pdf"))
    print(f"Found {len(pdf_files)} PDF files")
    
    for pdf_file in pdf_files:
        print(f"Processing: {pdf_file.name}")
        text = extract_text_from_pdf(pdf_file)
        
        if text and text.strip():  # Only add if text is not empty
            doc = Document(
                page_content=text,
                metadata={
                    "source": str(pdf_file),
                    "filename": pdf_file.name,
                    "type": "pdf_document"
                }
            )
            pdf_documents.append(doc)
        else:
            print(f"Warning: No text extracted from {pdf_file.name}")
    
    return pdf_documents

# Ingest all PDFs
pdf_docs = ingest_pdfs_from_data_folder()
print(f"Successfully ingested {len(pdf_docs)} PDF documents")

Found 8 PDF files
Processing: VC -- Periodic Testing Procedures_MH3000 Hydraulic Controllers – ASME A17.1-2019  CSA B44-19 Safety Code for Elevators.pdf
Processing: MH3000_Manual.pdf
Error reading data/MH3000_Manual.pdf: cannot open empty document
Processing: MH3000 Temporary Run Manual.pdf
Processing: VC--MVFAC-3000 User Manual (Rev 2_22).pdf
Processing: IP-8700 Installation.pdf
Processing: MH2000 to MH3000 Hardware Conversion Instructi.pdf
Processing: VC ETSD Installation & Operating Instructions.pdf
Processing: VC High Speed Counter Setup.pdf
Successfully ingested 7 PDF documents


In [4]:
df = pd.read_excel('10yearsdata.xls')
print("Data shape:", df.shape)

WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
Data shape: (25969, 22)


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25969 entries, 0 to 25968
Data columns (total 22 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   CaseID                 25969 non-null  int64         
 1   Job_Number             25969 non-null  str           
 2   Job_Name               25959 non-null  str           
 3   Case_Date              25969 non-null  datetime64[us]
 4   Case_Caller            25809 non-null  str           
 5   Case_Company           25672 non-null  str           
 6   Case_Phone             25413 non-null  float64       
 7   Case_Problem           25418 non-null  str           
 8   Case_Resolution_Notes  994 non-null    str           
 9   Case_Engineer          25220 non-null  str           
 10  Category               1050 non-null   str           
 11  Product_Code           25531 non-null  str           
 12  Actual_Ship            25395 non-null  datetime64[us]
 13  Time Receive

In [6]:
df.describe()

,CaseID,Case_Date,Case_Phone,Actual_Ship,ReportBeginDate,ReportEndDate
count,25969.000000,25969,2.541300e+04,25395,25969,25969
mean,13106.328738,2022-03-09 23:50:37.729600,5.822277e+09,2015-08-15 00:55:13.786178,2015-01-01 00:00:00,2025-12-29 00:00:00
min,32.000000,2018-02-13 00:00:00,0.000000e+00,1985-05-22 00:00:00,2015-01-01 00:00:00,2025-12-29 00:00:00
25%,6592.000000,2020-05-05 00:00:00,4.033691e+09,2010-08-18 00:00:00,2015-01-01 00:00:00,2025-12-29 00:00:00
50%,13111.000000,2022-04-21 00:00:00,6.063316e+09,2018-07-11 00:00:00,2015-01-01 00:00:00,2025-12-29 00:00:00
75%,19619.000000,2024-01-19 00:00:00,8.043392e+09,2021-09-09 00:00:00,2015-01-01 00:00:00,2025-12-29 00:00:00
max,26197.000000,2025-12-29 00:00:00,9.897150e+09,2025-12-18 00:00:00,2015-01-01 00:00:00,2025-12-29 00:00:00
std,7526.411136,NaN,2.335430e+09,NaN,NaN,NaN


In [7]:
df.isnull().sum()

CaseID                       0
Job_Number                   0
Job_Name                    10
Case_Date                    0
Case_Caller                160
Case_Company               297
Case_Phone                 556
Case_Problem               551
Case_Resolution_Notes    24975
Case_Engineer              749
Category                 24919
Product_Code               438
Actual_Ship                574
Time Received            24976
Time Call Started        24956
Category3                25058
Severity                 25383
Parts                    25897
Takeaway                 25907
ReportBeginDate              0
ReportEndDate                0
EmployeeName               745
dtype: int64

In [8]:
df = df[['CaseID', 'Job_Name', 'Case_Problem', 'Case_Resolution_Notes']]
df.head(10)

,CaseID,Job_Name,Case_Problem,Case_Resolution_Notes
0,32,VICTORIAN-ELEV #2,PROBEM GETTING OUT OF FIRE SVC.,NaN
1,33,HAMPTON INN EL-PE1,CAR RUNNING ON INSPECTION - ON CAR 2 SEVERAL F...,NaN
2,35,TARGET SALES-SMA,L4 LIGHT ON,NaN
3,41,OLD NATIONAL BANK E-2,TEST FOR EMERGENCY TERMINAL STOPPING DEVISE,NaN
4,42,VSU-HUNTER MCDANIEL HALL #2,CODES ON CHANGING DOOR TIMES,NaN
5,44,A.H.R.C. QUEENS #2,CAR IS NOT GOING DOWN,NaN
6,45,U OF AL-MAL MOORE DINING EL#A,NEED SOMEONE TO EMAIL QUICK STARTER QUIDE TO H...,NaN
7,46,240 CENTRAL ST.,CANT FIND FUSE #6,NaN
8,47,2145 SOUTH STREET,MISSING BUTTONS IN THE COP FOR REAR BUTTONS,NaN
9,48,SPY-ID TEST & EVALUATION,SELECTOR PROBLEM W/LEVEL UP & LEVEL DOWN AT SA...,NaN


In [9]:
print("Shape of the data:", df.shape)

Shape of the data: (25969, 4)


In [10]:
# Comprehensive data cleaning - remove all rows with empty data
print("Original data shape:", df.shape)
print("Original null counts:")
print(df.isnull().sum())

# Remove rows where ALL text columns are empty/NaN
text_columns = ['Job_Name', 'Case_Problem', 'Case_Resolution_Notes']
df_clean = df.dropna(subset=text_columns, how='all')

# Remove rows where essential fields are empty
df_clean = df_clean.dropna(subset=['Job_Name'])  # Must have job name
df_clean = df_clean.dropna(subset=['Case_Problem'])  # Must have problem description

# Fill missing resolution notes with placeholder
df_clean['Case_Resolution_Notes'] = df_clean['Case_Resolution_Notes'].fillna('No resolution recorded')

# Remove rows with empty strings after stripping whitespace
for col in text_columns:
    df_clean = df_clean[df_clean[col].astype(str).str.strip() != '']

# Reset index after cleaning
df_clean = df_clean.reset_index(drop=True)

print(f"\nCleaned data shape: {df_clean.shape}")
print(f"Removed {df.shape[0] - df_clean.shape[0]} rows with empty data")
print("\nNull counts after cleaning:")
print(df_clean.isnull().sum())

# Update df to use cleaned data
df = df_clean

Original data shape: (25969, 4)
Original null counts:
CaseID                       0
Job_Name                    10
Case_Problem               551
Case_Resolution_Notes    24975
dtype: int64

Cleaned data shape: (25409, 4)
Removed 560 rows with empty data

Null counts after cleaning:
CaseID                   0
Job_Name                 0
Case_Problem             0
Case_Resolution_Notes    0
dtype: int64


In [11]:
# Keep rows with at least problem description, fill missing resolutions
df = df.dropna(subset=['Job_Name'])  # Must have problem description
df = df.dropna(subset=['Case_Problem'])  # Must have problem description
df['Case_Resolution_Notes'] = df['Case_Resolution_Notes'].fillna('No resolution recorded')

In [12]:
# Initialize and use the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Create documents from Excel data
excel_documents = []
for idx, row in df.iterrows():
    combined_text = f"Problem: {row['Case_Problem']}\n\nResolution: {row['Case_Resolution_Notes']}"
    
    doc = Document(
        page_content=combined_text,
        metadata={
            "CaseID": row['CaseID'],
            "Job_Name": row['Job_Name'],
            "source": "excel_data",
            "type": "case_record"
        }
    )
    excel_documents.append(doc)

# Combine Excel documents and PDF documents
all_documents = excel_documents + pdf_docs

print(f"Total documents to process: {len(all_documents)}")
print(f"- Excel case records: {len(excel_documents)}")
print(f"- PDF documents: {len(pdf_docs)}")

# Split all documents into chunks
chunks = text_splitter.split_documents(all_documents)
print(f"Created {len(chunks)} chunks from {len(all_documents)} documents")

Total documents to process: 25416
- Excel case records: 25409
- PDF documents: 7
Created 25838 chunks from 25416 documents


In [13]:
# Initialize and use the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)
 
# Create documents and split them
documents = []
for idx, row in df.iterrows():
    combined_text = f"Problem: {row['Case_Problem']}\n\nResolution: {row['Case_Resolution_Notes']}"
    
    doc = Document(
        page_content=combined_text,
        metadata={
            "CaseID": row['CaseID'],
            "Job_Name": row['Job_Name']
        }
    )
    documents.append(doc)
 
# This line uses the text_splitter and removes the warning
chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks from {len(documents)} documents")

Created 25420 chunks from 25409 documents


In [14]:
# Create prompt template
template = """You are an elevator technician assistant. Use the following retrieved documents to answer the user's question.
The documents include both historical case records and technical manual PDFs. If no relevant information is found, respond with "No records found in the database."

Retrieved Documents:
{context}

User Question: {question}

Provide a helpful answer based strictly on the retrieved documents. For case records, include the CaseID and Job_Name as citations. For PDF documents, include the filename as citation."""

In [16]:
# Initialize embeddings using HuggingFace Hub (alternative approach)
from dotenv import load_dotenv
load_dotenv()

# Try different import approaches
try:
    from langchain_huggingface import HuggingFaceEndpointEmbeddings
    embeddings = HuggingFaceEndpointEmbeddings(
        huggingfacehub_api_token=os.getenv("HF_TOKEN"),
        model="sentence-transformers/all-MiniLM-L6-v2"
    )
    print("✅ Using HuggingFaceEndpointEmbeddings")
except ImportError as e:
    print(f"❌ HuggingFaceEndpointEmbeddings failed: {e}")
    try:
        from langchain_community.embeddings import HuggingFaceEmbeddings
        embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2",
            model_kwargs={'device': 'cpu'},
            encode_kwargs={'normalize_embeddings': True}
        )
        print("✅ Using HuggingFaceEmbeddings (local)")
    except ImportError as e2:
        print(f"❌ HuggingFaceEmbeddings failed: {e2}")
        raise ImportError("Both HuggingFace embedding methods failed")

# Create FAISS vector store
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

# Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # Return top 3 most relevant chunks
)

✅ Using HuggingFaceEndpointEmbeddings


In [17]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Create prompt template
template = """You are an elevator technician assistant. Use the following retrieved documents to answer the user's question.
The documents include both historical case records and technical manual PDFs. If no relevant information is found, respond with "No records found in the database."

Retrieved Documents:
{context}

User Question: {question}

Provide a helpful answer based strictly on retrieved documents. For case records, include CaseID and Job_Name as citations. For PDF documents, include the filename as citation."""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

# Create LCEL chain
def format_docs(docs):
    formatted_docs = []
    for doc in docs:
        doc_type = doc.metadata.get('type', 'unknown')
        if doc_type == 'case_record':
            formatted_docs.append(f"CaseID: {doc.metadata['CaseID']}, Job_Name: {doc.metadata['Job_Name']}\n{doc.page_content}")
        elif doc_type == 'pdf_document':
            formatted_docs.append(f"PDF: {doc.metadata['filename']}\n{doc.page_content}")
        else:
            formatted_docs.append(f"Document: {doc.metadata}\n{doc.page_content}")
    return "\n\n".join(formatted_docs)

# Check if retriever is defined before creating chain
try:
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    print("✅ RAG chain created successfully")
except NameError:
    print("❌ Error: 'retriever' not defined. Please run the embeddings cell first.")
    print("💡 Make sure cell 14 (embeddings) runs before this cell.")
    rag_chain = None

✅ RAG chain created successfully


In [18]:
# Comprehensive test questions to validate the RAG system
test_questions = [
    # Technical manual questions (should come from PDFs)
    "How do I perform a hardware conversion from MH2000 to MH3000?",
    "What are the installation procedures for IP-8700?",
    "What are the temporary run procedures for MH3000?",
    "How do I set up the high speed counter?",
    "What are the periodic testing procedures for MH3000 hydraulic controllers?",
    "How do I reset UCM board faults?",
    "What are the duplex communication requirements?",
    
    # Case record questions (should come from Excel data)
    "How do I fix a speed deviation fault on the MH3000?",
    "What causes elevator slowdown issues?",
    "How do I troubleshoot MH3000 controller faults?",
    "What are common problems with elevator door operations?",
    "How do I fix elevator emergency terminal stopping devices?",
    
    # Mixed questions (could come from both sources)
    "What safety procedures should I follow when working on MH3000 controllers?",
    "How do I diagnose elevator communication problems?",
    "What are the common fault codes for MH3000?",
    
    # Edge cases
    "How do I install an elevator in a residential building?",
    "What are the maintenance schedules for elevator components?",
    "How do I fix a specific error code E-1234?",
    
    # Specific technical queries
    "What are the voltage requirements for MH3000?",
    "How do I configure the MVFAC-3000 parameters?",
    "What are the ASME A17.1-2019 safety code requirements?"
]

print("=== RAG System Validation Test Questions ===\n")
for i, question in enumerate(test_questions, 1):
    print(f"{i:2d}. {question}")

print(f"\nTotal test questions: {len(test_questions)}")
print("\nCategories:")
print(f"- Technical Manual Questions: 7")
print(f"- Case Record Questions: 5") 
print(f"- Mixed Source Questions: 3")
print(f"- Edge Cases: 3")
print(f"- Specific Technical Queries: 3")

=== RAG System Validation Test Questions ===

 1. How do I perform a hardware conversion from MH2000 to MH3000?
 2. What are the installation procedures for IP-8700?
 3. What are the temporary run procedures for MH3000?
 4. How do I set up the high speed counter?
 5. What are the periodic testing procedures for MH3000 hydraulic controllers?
 6. How do I reset UCM board faults?
 7. What are the duplex communication requirements?
 8. How do I fix a speed deviation fault on the MH3000?
 9. What causes elevator slowdown issues?
10. How do I troubleshoot MH3000 controller faults?
11. What are common problems with elevator door operations?
12. How do I fix elevator emergency terminal stopping devices?
13. What safety procedures should I follow when working on MH3000 controllers?
14. How do I diagnose elevator communication problems?
15. What are the common fault codes for MH3000?
16. How do I install an elevator in a residential building?
17. What are the maintenance schedules for elevator c

In [19]:
# Quick test function for individual questions
def quick_test(question):
    """Test a single question quickly"""
    print(f"Question: {question}")
    print("-" * 50)
    
    try:
        response = rag_chain.invoke(question)
        print(f"Response:\n{response}")
        
        # Show retrieved documents for analysis
        retrieved_docs = retriever.invoke(question)
        print(f"\nRetrieved {len(retrieved_docs)} documents:")
        for i, doc in enumerate(retrieved_docs):
            doc_type = doc.metadata.get('type', 'unknown')
            if doc_type == 'case_record':
                source = f"Case {doc.metadata.get('CaseID', 'N/A')}"
            elif doc_type == 'pdf_document':
                source = f"PDF: {doc.metadata.get('filename', 'N/A')}"
            else:
                source = str(doc.metadata)
            print(f"  {i+1}. [{doc_type}] {source}")
        
    except Exception as e:
        print(f"Error: {e}")

# Example usage - uncomment to test
# quick_test("How do I perform MH2000 to MH3000 hardware conversion?")
# quick_test("What causes elevator slowdown issues?")

In [20]:
# RAG Accuracy Measurement Framework
import time
from typing import List, Dict, Tuple
import numpy as np
from sklearn.metrics import classification_report
import re

class RAGEvaluator:
    def __init__(self, rag_chain, retriever):
        self.rag_chain = rag_chain
        self.retriever = retriever
        self.evaluation_results = []
    
    def evaluate_retrieval_quality(self, question: str, expected_keywords: List[str] = None) -> Dict:
        """Evaluate retrieval component quality"""
        retrieved_docs = self.retriever.invoke(question)
        
        metrics = {
            'num_retrieved_docs': len(retrieved_docs),
            'doc_types': [doc.metadata.get('type', 'unknown') for doc in retrieved_docs],
            'has_relevant_content': False,
            'keyword_matches': 0
        }
        
        # Check if expected keywords are found
        if expected_keywords:
            for doc in retrieved_docs:
                doc_text = doc.page_content.lower()
                for keyword in expected_keywords:
                    if keyword.lower() in doc_text:
                        metrics['keyword_matches'] += 1
                        metrics['has_relevant_content'] = True
        
        return metrics
    
    def evaluate_response_quality(self, question: str, response: str) -> Dict:
        """Evaluate response quality metrics"""
        metrics = {
            'response_length': len(response),
            'has_citations': bool(re.search(r'CaseID:\s*\d+|PDF:\s*\w+', response)),
            'is_helpful': not bool(re.search(r'no records found|no information|not found', response, re.IGNORECASE)),
            'is_specific': len(response.split()) > 20,
            'has_technical_terms': bool(re.search(r'MH\d+|controller|elevator|fault|installation', response, re.IGNORECASE))
        }
        
        return metrics
    
    def evaluate_latency(self, question: str) -> Dict:
        """Measure response latency"""
        start_time = time.time()
        response = self.rag_chain.invoke(question)
        end_time = time.time()
        
        return {
            'latency_seconds': end_time - start_time,
            'response': response
        }
    
    def comprehensive_evaluation(self, test_cases: List[Dict]) -> Dict:
        """Run comprehensive evaluation on test cases"""
        results = {
            'total_tests': len(test_cases),
            'retrieval_metrics': [],
            'response_metrics': [],
            'latency_metrics': [],
            'overall_scores': {}
        }
        
        for i, test_case in enumerate(test_cases):
            question = test_case['question']
            expected_keywords = test_case.get('expected_keywords', [])
            category = test_case.get('category', 'general')
            
            print(f"Evaluating test {i+1}/{len(test_cases)}: {category}")
            
            # Evaluate retrieval
            retrieval_metrics = self.evaluate_retrieval_quality(question, expected_keywords)
            results['retrieval_metrics'].append(retrieval_metrics)
            
            # Evaluate response
            latency_result = self.evaluate_latency(question)
            response = latency_result['response']
            response_metrics = self.evaluate_response_quality(question, response)
            response_metrics['latency'] = latency_result['latency_seconds']
            results['response_metrics'].append(response_metrics)
            results['latency_metrics'].append(latency_result['latency_seconds'])
        
        # Calculate overall scores
        results['overall_scores'] = self.calculate_overall_scores(results)
        
        return results
    
    def calculate_overall_scores(self, results: Dict) -> Dict:
        """Calculate overall performance scores"""
        retrieval_scores = []
        response_scores = []
        
        # Retrieval scores
        for metric in results['retrieval_metrics']:
            score = 0
            if metric['num_retrieved_docs'] > 0:
                score += 0.3
            if metric['has_relevant_content']:
                score += 0.4
            if metric['keyword_matches'] > 0:
                score += 0.3
            retrieval_scores.append(score)
        
        # Response quality scores
        for metric in results['response_metrics']:
            score = 0
            if metric['is_helpful']:
                score += 0.3
            if metric['has_citations']:
                score += 0.2
            if metric['is_specific']:
                score += 0.3
            if metric['has_technical_terms']:
                score += 0.2
            response_scores.append(score)
        
        return {
            'avg_retrieval_score': np.mean(retrieval_scores) if retrieval_scores else 0,
            'avg_response_score': np.mean(response_scores) if response_scores else 0,
            'avg_latency': np.mean(results['latency_metrics']) if results['latency_metrics'] else 0,
            'overall_score': (np.mean(retrieval_scores) + np.mean(response_scores)) / 2 if retrieval_scores and response_scores else 0
        }

# Initialize evaluator
evaluator = RAGEvaluator(rag_chain, retriever)

In [21]:
# Define comprehensive test cases with expected keywords
comprehensive_test_cases = [
    {
        'question': 'How do I perform MH2000 to MH3000 hardware conversion?',
        'category': 'technical_manual',
        'expected_keywords': ['MH2000', 'MH3000', 'conversion', 'hardware']
    },
    {
        'question': 'What are the installation procedures for IP-8700?',
        'category': 'technical_manual',
        'expected_keywords': ['IP-8700', 'installation', 'procedures']
    },
    {
        'question': 'How do I fix a speed deviation fault on the MH3000?',
        'category': 'case_records',
        'expected_keywords': ['speed', 'deviation', 'fault', 'MH3000']
    },
    {
        'question': 'What causes elevator slowdown issues?',
        'category': 'case_records',
        'expected_keywords': ['slowdown', 'elevator', 'issues']
    },
    {
        'question': 'What are the periodic testing procedures for MH3000 hydraulic controllers?',
        'category': 'technical_manual',
        'expected_keywords': ['periodic', 'testing', 'MH3000', 'hydraulic']
    },
    {
        'question': 'How do I troubleshoot MH3000 controller faults?',
        'category': 'mixed_source',
        'expected_keywords': ['troubleshoot', 'MH3000', 'controller', 'faults']
    },
    {
        'question': 'What are the ASME A17.1-2019 safety code requirements?',
        'category': 'technical_manual',
        'expected_keywords': ['ASME', 'A17.1-2019', 'safety', 'code']
    },
    {
        'question': 'How do I reset UCM board faults?',
        'category': 'technical_manual',
        'expected_keywords': ['UCM', 'board', 'faults', 'reset']
    }
]

print("Comprehensive Test Cases Defined:")
for i, test in enumerate(comprehensive_test_cases, 1):
    print(f"{i}. [{test['category']}] {test['question']}")
    print(f"   Keywords: {test['expected_keywords']}")
print()

Comprehensive Test Cases Defined:
1. [technical_manual] How do I perform MH2000 to MH3000 hardware conversion?
   Keywords: ['MH2000', 'MH3000', 'conversion', 'hardware']
2. [technical_manual] What are the installation procedures for IP-8700?
   Keywords: ['IP-8700', 'installation', 'procedures']
3. [case_records] How do I fix a speed deviation fault on the MH3000?
   Keywords: ['speed', 'deviation', 'fault', 'MH3000']
4. [case_records] What causes elevator slowdown issues?
   Keywords: ['slowdown', 'elevator', 'issues']
5. [technical_manual] What are the periodic testing procedures for MH3000 hydraulic controllers?
   Keywords: ['periodic', 'testing', 'MH3000', 'hydraulic']
6. [mixed_source] How do I troubleshoot MH3000 controller faults?
   Keywords: ['troubleshoot', 'MH3000', 'controller', 'faults']
7. [technical_manual] What are the ASME A17.1-2019 safety code requirements?
   Keywords: ['ASME', 'A17.1-2019', 'safety', 'code']
8. [technical_manual] How do I reset UCM board faults?


In [22]:
# Run comprehensive evaluation
print("=== Starting Comprehensive RAG Evaluation ===\n")

# Run evaluation
evaluation_results = evaluator.comprehensive_evaluation(comprehensive_test_cases)

# Display results
print("=== EVALUATION RESULTS ===\n")

print("📊 OVERALL SCORES:")
scores = evaluation_results['overall_scores']
print(f"Retrieval Score: {scores['avg_retrieval_score']:.2f}/1.00")
print(f"Response Score: {scores['avg_response_score']:.2f}/1.00")
print(f"Overall Score: {scores['overall_score']:.2f}/1.00")
print(f"Average Latency: {scores['avg_latency']:.2f} seconds")

print("\n📈 DETAILED METRICS:")

# Retrieval analysis
retrieval_docs = [m['num_retrieved_docs'] for m in evaluation_results['retrieval_metrics']]
retrieval_relevant = [m['has_relevant_content'] for m in evaluation_results['retrieval_metrics']]
print(f"Retrieval: Avg docs retrieved: {np.mean(retrieval_docs):.1f}")
print(f"Retrieval: Relevant content rate: {np.mean(retrieval_relevant)*100:.1f}%")

# Response analysis
helpful_responses = [m['is_helpful'] for m in evaluation_results['response_metrics']]
cited_responses = [m['has_citations'] for m in evaluation_results['response_metrics']]
specific_responses = [m['is_specific'] for m in evaluation_results['response_metrics']]
technical_responses = [m['has_technical_terms'] for m in evaluation_results['response_metrics']]

print(f"Response: Helpful rate: {np.mean(helpful_responses)*100:.1f}%")
print(f"Response: Citation rate: {np.mean(cited_responses)*100:.1f}%")
print(f"Response: Specific rate: {np.mean(specific_responses)*100:.1f}%")
print(f"Response: Technical content rate: {np.mean(technical_responses)*100:.1f}%")

print("\n🎯 PERFORMANCE BREAKDOWN:")
print("Score Interpretation:")
print("0.80-1.00: Excellent")
print("0.60-0.79: Good") 
print("0.40-0.59: Fair")
print("0.20-0.39: Poor")
print("0.00-0.19: Very Poor")

=== Starting Comprehensive RAG Evaluation ===

Evaluating test 1/8: technical_manual
Evaluating test 2/8: technical_manual
Evaluating test 3/8: case_records
Evaluating test 4/8: case_records
Evaluating test 5/8: technical_manual
Evaluating test 6/8: mixed_source
Evaluating test 7/8: technical_manual
Evaluating test 8/8: technical_manual
=== EVALUATION RESULTS ===

📊 OVERALL SCORES:
Retrieval Score: 0.91/1.00
Response Score: 0.68/1.00
Overall Score: 0.79/1.00
Average Latency: 2.20 seconds

📈 DETAILED METRICS:
Retrieval: Avg docs retrieved: 3.0
Retrieval: Relevant content rate: 87.5%
Response: Helpful rate: 50.0%
Response: Citation rate: 25.0%
Response: Specific rate: 100.0%
Response: Technical content rate: 87.5%

🎯 PERFORMANCE BREAKDOWN:
Score Interpretation:
0.80-1.00: Excellent
0.60-0.79: Good
0.40-0.59: Fair
0.20-0.39: Poor
0.00-0.19: Very Poor


In [23]:
# Additional RAG Evaluation Methods

def manual_evaluation_checklist():
    """Manual evaluation checklist for RAG systems"""
    print("=== MANUAL EVALUATION CHECKLIST ===\n")
    
    checklist = {
        "Retrieval Quality": [
            "✓ Are retrieved documents relevant to the query?",
            "✓ Is there a good mix of PDF and case record sources?",
            "✓ Are technical terms properly matched?",
            "✓ Is the retrieval depth appropriate (not too shallow/deep)?"
        ],
        "Response Quality": [
            "✓ Does the answer directly address the user's question?",
            "✓ Are sources properly cited?",
            "✓ Is the technical information accurate?",
            "✓ Is the response appropriately detailed?",
            "✓ Does it avoid hallucination/making up information?"
        ],
        "System Performance": [
            "✓ Are response times acceptable (<5 seconds)?",
            "✓ Does the system handle different question types?",
            "✓ Are edge cases handled gracefully?",
            "✓ Is the system consistent across similar queries?"
        ],
        "Coverage Analysis": [
            "✓ Are all PDF documents accessible and searchable?",
            "✓ Are case records properly indexed?",
            "✓ Is there good coverage across different topics?",
            "✓ Are document types properly identified?"
        ]
    }
    
    for category, items in checklist.items():
        print(f"🔍 {category}:")
        for item in items:
            print(f"  {item}")
        print()

def automated_metrics_summary():
    """Summary of automated RAG metrics"""
    print("=== AUTOMATED RAG METRICS ===\n")
    
    metrics = {
        "Retrieval Metrics": [
            "Precision@K: Fraction of retrieved docs that are relevant",
            "Recall@K: Fraction of relevant docs that are retrieved",
            "MRR (Mean Reciprocal Rank): Average reciprocal rank of first relevant doc",
            "Coverage: Percentage of queries that return any relevant docs"
        ],
        "Generation Metrics": [
            "Faithfulness: Does response stick to retrieved context?",
            "Answer Relevancy: Is response relevant to the question?",
            "Citation Accuracy: Are citations correct and complete?",
            "Response Length: Appropriate detail level"
        ],
        "End-to-End Metrics": [
            "Latency: Total response time",
            "Throughput: Queries per second",
            "Success Rate: Percentage of successful responses",
            "User Satisfaction: Subjective quality ratings"
        ]
    }
    
    for category, items in metrics.items():
        print(f"📊 {category}:")
        for item in items:
            print(f"  • {item}")
        print()

# Run manual checklist
manual_evaluation_checklist()

# Show automated metrics summary
automated_metrics_summary()

print("💡 RECOMMENDATION:")
print("Combine automated metrics with human evaluation for comprehensive RAG assessment.")
print("Automated metrics provide scalability, while human evaluation ensures quality.")

=== MANUAL EVALUATION CHECKLIST ===

🔍 Retrieval Quality:
  ✓ Are retrieved documents relevant to the query?
  ✓ Is there a good mix of PDF and case record sources?
  ✓ Are technical terms properly matched?
  ✓ Is the retrieval depth appropriate (not too shallow/deep)?

🔍 Response Quality:
  ✓ Does the answer directly address the user's question?
  ✓ Are sources properly cited?
  ✓ Is the technical information accurate?
  ✓ Is the response appropriately detailed?
  ✓ Does it avoid hallucination/making up information?

🔍 System Performance:
  ✓ Are response times acceptable (<5 seconds)?
  ✓ Does the system handle different question types?
  ✓ Are edge cases handled gracefully?
  ✓ Is the system consistent across similar queries?

🔍 Coverage Analysis:
  ✓ Are all PDF documents accessible and searchable?
  ✓ Are case records properly indexed?
  ✓ Is there good coverage across different topics?
  ✓ Are document types properly identified?

=== AUTOMATED RAG METRICS ===

📊 Retrieval Metrics

In [24]:
# Run comprehensive test suite
def run_test_suite(questions, max_questions=5):
    """Run a subset of test questions and evaluate responses"""
    print(f"=== Running {max_questions} Test Questions ===\n")
    
    results = []
    for i, question in enumerate(questions[:max_questions], 1):
        print(f"Question {i}: {question}")
        print("-" * 60)
        
        try:
            response = rag_chain.invoke(question)
            print(f"Response: {response}")
            
            # Evaluate response quality
            if "No records found" in response:
                status = "❌ No relevant information found"
            elif len(response) < 50:
                status = "⚠️  Very short response"
            else:
                status = "✅ Good response"
            
            results.append({
                "question": question,
                "response": response,
                "status": status
            })
            
        except Exception as e:
            print(f"Error: {e}")
            results.append({
                "question": question,
                "response": f"Error: {e}",
                "status": "❌ Error occurred"
            })
        
        print(f"Status: {status}")
        print("=" * 80 + "\n")
    
    return results

# Run the test suite
test_results = run_test_suite(test_questions, max_questions=5)

# Summary
print("=== Test Summary ===")
for result in test_results:
    print(f"{result['status']}: {result['question'][:60]}...")

=== Running 5 Test Questions ===

Question 1: How do I perform a hardware conversion from MH2000 to MH3000?
------------------------------------------------------------
Response: To perform a hardware conversion from MH2000 to MH3000, you should follow the instructions provided on the sheet titled "MH2000 Conversion instructions" and sheet 1, as referenced in the resolution for CaseID 25451, BERNARDS ELEMENTARY [1]. 

Unfortunately, the exact details of these instructions are not available in the retrieved documents. However, it is confirmed that such instructions exist and have been used in the past for a successful conversion.

For further guidance, you may want to review the technical manual or documentation that accompanies the MH3000 Upgrade Kit.

References:
[1] CaseID 25451, BERNARDS ELEMENTARY
Status: ✅ Good response

Question 2: What are the installation procedures for IP-8700?
------------------------------------------------------------
Response: No records found in the datab